# Type-Safe Options with `pyvis.types`

PyVis provides 46 Python dataclasses that cover 100% of the vis-network configuration surface. Instead of writing raw JSON strings, you get IDE autocompletion, type checking, and readable configuration.

**What you'll learn:**
- Configuring node appearance with `NodeOptions`
- Styling edges with `EdgeOptions`
- Choosing physics solvers with `PhysicsOptions`
- Hierarchical layouts with `LayoutOptions`
- Interaction settings with `InteractionOptions`
- Combining everything with `NetworkOptions`

In [ ]:
from pyvis.network import Network
from pyvis.types import (
    NetworkOptions, NodeOptions, EdgeOptions, PhysicsOptions,
    BarnesHut, ForceAtlas2Based, Repulsion,
    LayoutOptions, HierarchicalLayout,
    InteractionOptions, Font, Shadow, EdgeSmooth,
    EdgeArrows, ArrowConfig, NodeColor, ColorHighlight,
)
import networkx as nx

## 1. Node Styling with `NodeOptions`

Configure the default appearance of all nodes: shape, size, font, shadow, and color with highlight states.

In [ ]:
options = NetworkOptions(
    nodes=NodeOptions(
        shape="dot",
        size=20,
        font=Font(size=14, color="#333333", face="arial"),
        shadow=Shadow(enabled=True, size=10),
        color=NodeColor(
            background="#6366f1",
            border="#4f46e5",
            highlight=ColorHighlight(background="#818cf8", border="#6366f1"),
        ),
    ),
)

net = Network(notebook=True, cdn_resources="in_line", height="400px")
net.set_options(options)
for i in range(1, 8):
    net.add_node(i, label=f"Node {i}")
for i in range(1, 7):
    net.add_edge(i, i + 1)
net.add_edge(7, 1)
net.show("03_nodes.html")

## 2. Edge Styling with `EdgeOptions`

Control edge color, width, arrows, and curve type. Smooth edges can be `dynamic`, `continuous`, `curvedCW`, `curvedCCW`, and more.

In [ ]:
options = NetworkOptions(
    edges=EdgeOptions(
        color="#94a3b8",
        width=2,
        arrows=EdgeArrows(to=ArrowConfig(enabled=True, scaleFactor=0.8)),
        smooth=EdgeSmooth(enabled=True, type="curvedCW", roundness=0.2),
        shadow=Shadow(enabled=True),
    ),
)

net = Network(notebook=True, cdn_resources="in_line", height="400px", directed=True)
net.set_options(options)
for i in range(1, 6):
    net.add_node(i, label=f"Step {i}", color="#6366f1")
for i in range(1, 5):
    net.add_edge(i, i + 1)
net.show("03_edges.html")

## 3. Physics Solvers

vis-network includes several physics engines. Use `PhysicsOptions` to select and configure them.

### Barnes-Hut (default)
Fast approximation of n-body simulation. Good for most graphs.

In [ ]:
G = nx.barabasi_albert_graph(25, 2, seed=42)

options = NetworkOptions(
    physics=PhysicsOptions(
        solver="barnesHut",
        barnesHut=BarnesHut(
            gravitationalConstant=-3000,
            springLength=150,
            springConstant=0.04,
            damping=0.09,
        ),
    ),
)

net = Network(notebook=True, cdn_resources="in_line", height="400px")
net.set_options(options)
net.from_nx(G)
net.show("03_barnes_hut.html")

### Force Atlas 2
Community-detection-friendly layout. Nodes in dense clusters pull together.

In [ ]:
options = NetworkOptions(
    physics=PhysicsOptions(
        solver="forceAtlas2Based",
        forceAtlas2Based=ForceAtlas2Based(
            gravitationalConstant=-50,
            centralGravity=0.01,
            springLength=100,
            damping=0.4,
        ),
    ),
)

net = Network(notebook=True, cdn_resources="in_line", height="400px")
net.set_options(options)
net.from_nx(G)
net.show("03_force_atlas.html")

### Repulsion
Simple repulsion model. Nodes push each other away, edges act as springs.

In [ ]:
options = NetworkOptions(
    physics=PhysicsOptions(
        solver="repulsion",
        repulsion=Repulsion(
            nodeDistance=150,
            centralGravity=0.1,
            springLength=200,
        ),
    ),
)

net = Network(notebook=True, cdn_resources="in_line", height="400px")
net.set_options(options)
net.from_nx(G)
net.show("03_repulsion.html")

## 4. Hierarchical Layout

Use `LayoutOptions` with `HierarchicalLayout` for tree-like structures. Directions: `"UD"` (up-down), `"DU"`, `"LR"`, `"RL"`.

In [ ]:
options = NetworkOptions(
    layout=LayoutOptions(
        hierarchical=HierarchicalLayout(
            enabled=True,
            direction="UD",
            sortMethod="directed",
            levelSeparation=100,
        ),
    ),
    physics=PhysicsOptions(enabled=False),
)

net = Network(notebook=True, cdn_resources="in_line", height="500px", directed=True)
net.set_options(options)

# Build an org chart
net.add_node("CEO", label="CEO", color="#f87171", size=30)
for dept in ["Engineering", "Marketing", "Sales"]:
    net.add_node(dept, label=dept, color="#fbbf24", size=25)
    net.add_edge("CEO", dept)
for team, parent in [("Frontend", "Engineering"), ("Backend", "Engineering"),
                     ("SEO", "Marketing"), ("Social", "Marketing"),
                     ("Enterprise", "Sales"), ("SMB", "Sales")]:
    net.add_node(team, label=team, color="#34d399", size=20)
    net.add_edge(parent, team)
net.show("03_hierarchical.html")

## 5. Interaction Options

Enable hover effects, keyboard navigation, and navigation buttons.

In [ ]:
options = NetworkOptions(
    interaction=InteractionOptions(
        hover=True,
        tooltipDelay=100,
        navigationButtons=True,
        keyboard=True,
    ),
)

net = Network(notebook=True, cdn_resources="in_line", height="400px")
net.set_options(options)
for i in range(1, 11):
    net.add_node(i, label=f"Node {i}", title=f"Hover info for node {i}")
for i in range(1, 10):
    net.add_edge(i, i + 1)
net.add_edge(10, 1)
net.show("03_interaction.html")

## 6. Combining Everything with `NetworkOptions`

`NetworkOptions` brings together all sub-options into a single configuration object.

In [ ]:
options = NetworkOptions(
    nodes=NodeOptions(shape="dot", font=Font(size=12, color="#e2e8f0")),
    edges=EdgeOptions(color="#475569", width=1, smooth=True),
    physics=PhysicsOptions(
        solver="barnesHut",
        barnesHut=BarnesHut(gravitationalConstant=-2000),
    ),
    interaction=InteractionOptions(hover=True, tooltipDelay=200),
)

net = Network(notebook=True, cdn_resources="in_line", height="500px",
              bgcolor="#0f172a", font_color="#e2e8f0")
net.set_options(options)

G = nx.karate_club_graph()
net.from_nx(G)
neighbor_map = net.get_adj_list()
for node in net.nodes:
    degree = len(neighbor_map[node["id"]])
    node["title"] = f"Node {node['id']}, degree {degree}"
    node["value"] = degree
net.show("03_combined.html")

## Legacy vs. Typed Options

**Before (JSON string):**
```python
net.set_options('''var options = {
  "physics": {
    "barnesHut": {
      "gravitationalConstant": -3000,
      "springLength": 150
    }
  }
}''')
```

**After (typed dataclass):**
```python
from pyvis.types import NetworkOptions, PhysicsOptions, BarnesHut

net.set_options(NetworkOptions(
    physics=PhysicsOptions(
        barnesHut=BarnesHut(gravitationalConstant=-3000, springLength=150)
    )
))
```

The typed approach gives you IDE autocompletion, catches typos at construction time, and is more readable.

---
**Previous:** [02_networkx.ipynb](02_networkx.ipynb) | **Next:** [04_advanced.ipynb](04_advanced.ipynb)